In [88]:
import pandas as pd

In [89]:
shop_data = pd.read_csv("/kaggle/input/datasets/premraj25/shop-smart-dataset/shop_smart_ecommerce.csv")

In [90]:
shop_data.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [91]:
shop_data.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [92]:
shop_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [93]:
# Encoding 

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

feature = ["Month","VisitorType","Weekend", "Revenue"]

for col in feature:
    shop_data[col] = le.fit_transform(shop_data[col])

In [96]:
# Feature Engineering

shop_data["Time_Per_Page"] = shop_data["ProductRelated_Duration"] / shop_data["ProductRelated"]
shop_data["Time_Per_Page"] = shop_data["Time_Per_Page"].fillna(0) # Fix division by zero

shop_data["Total_Time_On_Site"] = shop_data["Administrative_Duration"] + shop_data["Informational_Duration"] + shop_data["ProductRelated_Duration"]

In [97]:
shop_data.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
Time_Per_Page              0
Total_Time_On_Site         0
dtype: int64

In [99]:
X = shop_data.drop(columns = ["OperatingSystems", "Revenue", "Browser"], axis = 1)
Y = shop_data["Revenue"]

In [100]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,
    test_size = 0.3,                                                
    random_state = 42
)

In [101]:
X_train.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,Region,TrafficType,VisitorType,Weekend,Time_Per_Page,Total_Time_On_Site
8077,0,0.0,0,0.0,5,81.083333,0.04,0.050000,0.0,0.0,1,1,2,0,0,16.216667,81.083333
1568,0,0.0,0,0.0,3,189.000000,0.00,0.066667,0.0,0.0,5,8,1,2,0,63.000000,189.000000
462,0,0.0,1,132.0,8,445.000000,0.00,0.014286,0.0,0.0,5,4,14,2,1,55.625000,577.000000
743,0,0.0,0,0.0,3,0.000000,0.20,0.200000,0.0,0.0,5,2,1,2,0,0.000000,0.000000
222,0,0.0,0,0.0,4,14.000000,0.10,0.150000,0.0,0.0,5,1,1,2,0,3.500000,14.000000


In [102]:
Y_train.head()

8077    0
1568    0
462     0
743     0
222     0
Name: Revenue, dtype: int64

In [103]:
# Decision Tree Model with Pre-pruning 

from sklearn.tree import DecisionTreeClassifier
dtc_model = DecisionTreeClassifier(max_depth=4)

dtc_model.fit(X_train, Y_train)

DecisionTreeClassifier(max_depth=4)

In [104]:
Y_pred = dtc_model.predict(X_test)
print(Y_pred[:10])
Y_test.head(10)

from sklearn.metrics import f1_score

print("F1 Score with Pre-pruning:",f1_score(Y_test,Y_pred))

[0 0 1 0 0 0 0 0 0 0]
F1 Score with Pre-pruning: 0.5926701570680628


In [105]:
# Post-pruning

dtc_full_tree = DecisionTreeClassifier(random_state = 42)
dtc_full_tree.fit(X_train, Y_train)

DecisionTreeClassifier(random_state=42)

In [106]:
path = dtc_full_tree.cost_complexity_pruning_path(X_train, Y_train)
ccp_alphas = path.ccp_alphas

In [107]:
# train our model for all alphas 

tree = []

for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state = 42, ccp_alpha = alpha)
    model.fit(X_train, Y_train)
    tree.append((model,alpha))

best_f1 = 0
best_alpha = 0

for model, alpha in tree:
    Y_pred = model.predict(X_test)
    cur_f1 = f1_score(Y_test,Y_pred)
    if cur_f1 > best_f1:
        best_f1 = cur_f1
        best_alpha = alpha

best_model = DecisionTreeClassifier(ccp_alpha=best_alpha)
best_model.fit(X_train, Y_train)

DecisionTreeClassifier(ccp_alpha=np.float64(0.010413172679856406))

In [108]:
Y_pred = best_model.predict(X_test)
print("F1 Score after Post-Pruning:",f1_score(Y_test,Y_pred))

F1 Score after Post-Pruning: 0.6604046242774566
